# Prepare data
This notebook serve the purpose of preparing the data for the training of the model. It includes dataset formatting to the right format, splitting the dataset, cleaning it by removing the cases that we marked as too damaged, computing the common foreground mask, and statistics about the training data that will be used for the data augmentation and the training of the model.

## CHASE dataset

In [ ]:
chase = AvailableDatasets.CHASE.value

chase_dataset_dir = chase.get_data_dir()
print(f"CHASE dataset directory: {chase_dataset_dir}")

In [ ]:
import json
from tqdm import tqdm
from PIL import Image
import torch
import numpy as np
import os

def process_all(main_dir, keep_annotation_id=1):

    src_dir = os.path.join(main_dir, "original")

    dst_img_dir = os.path.join(main_dir, "img")
    dst_gt_dir = os.path.join(main_dir, "gt")
    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_gt_dir, exist_ok=True)

    # --- List image and ground truth filenames ---
    all_files_list = os.listdir(src_dir)
    idx_list = []
    for f in tqdm(all_files_list):
        filepath = os.path.join(src_dir, f)

        underscores_list = f.split('_')
        i = int(underscores_list[1][:2])
        orientation = underscores_list[1][2]
        if orientation != 'L':
            i = 2 * i - 1
        else:
            i = 2 * i
        id_name = f"CHASE_{i:03d}"
        idx_list.append(id_name)

        if len(underscores_list) == 2: # is an img
            # --- Convert image (.jpg -> .png) ---
            dst_img_path = os.path.join(dst_img_dir, f"{id_name}.png")
            img = Image.open(filepath)
            img.save(dst_img_path)
        elif len(underscores_list) == 3: # is a gt
            annotation_id = int(underscores_list[2][0])
            if annotation_id != keep_annotation_id:
                continue
            # --- Already a png, just copy it to the right place with the correct name ---
            dst_gt_path = os.path.join(dst_gt_dir, f"{id_name}.png")
            gt = Image.open(filepath)
            gt.save(dst_gt_path)

    idx_list = list(set(idx_list)) # Remove duplicates
    return idx_list

chase_all_idx = process_all(chase_dataset_dir)

The CHASE dataset doesn't have official splits, so we create our own splits by randomly splitting the dataset into a training set (80%) and a test set (20%).

In [ ]:
from sklearn.model_selection import train_test_split

chase_train_idx, chase_test_idx = train_test_split(chase_all_idx, test_size=0.2, random_state=42)

chase_splits = {}
chase_splits['train'] = chase_train_idx
chase_splits['test'] = chase_test_idx

# === Save splits ===
chase_splits_filepath = os.path.join(chase_dataset_dir, "splits.json")
with open(chase_splits_filepath, 'w') as f:
    json.dump(chase_splits, f, indent=4)

In [ ]:
import numpy as np
from PIL import Image
from skimage.measure import label
import matplotlib.pyplot as plt

img_dir = os.path.join(chase_dataset_dir, 'img/')
img_paths = [os.path.join(img_dir, fname) for fname in os.listdir(img_dir) if fname.endswith('.png')]

threshold = 2
debug_i = 100

non_zero_masks = []
for i, img_path in enumerate(img_paths):
    if i >= debug_i:
        break

    img_gray = Image.open(img_path).convert("L")
    img_gray = np.array(img_gray)
    non_zero_mask = img_gray > threshold

    cc, num_cc = label(non_zero_mask, return_num=True, connectivity=2)
    cc_sizes = [np.sum(cc == i) for i in range(1, num_cc + 1)]
    max_cc_index = np.argmax(cc_sizes) + 1
    non_zero_mask = cc == max_cc_index

    non_zero_masks.append(non_zero_mask)

common_mask = np.logical_and.reduce(non_zero_masks)

In [ ]:
import matplotlib.pyplot as plt
import torch

plt.imshow(common_mask, cmap='gray')
plt.title(f"Common Mask for CHASE Dataset Between First {debug_i} Images")
plt.axis('off')
plt.show()

foreground_mask_dir = os.path.join(chase_dataset_dir, "foreground_masks")
os.makedirs(foreground_mask_dir, exist_ok=True)

torch.save(torch.from_numpy(common_mask), os.path.join(foreground_mask_dir, "CHASE.pt"))

In [ ]:
from image_segmentation.data import ImageDataset, ImageDatamodule

chase_dataset = ImageDataset(data_dir=chase_dataset_dir, transforms=None)
chase_datamodule = ImageDatamodule(chase_dataset, 
                             split_file_path=chase_splits_filepath,
                             train_split_name='train',
                             val_split_ratio=0.2,
                             train_transforms=None,
                             val_transforms=None,
                             test_transforms=None,
                             num_workers=0,
                             train_batch_size=4,
                             val_batch_size=1,
                             seed=42,
                             shuffle_train=True)
chase_datamodule.setup()

In [ ]:
# We use a FOV of 30 degrees for the CHASE dataset, as specified in the dataset documentation.
stats = chase_dataset.get_dataset_stats(split_name='train', split_indices=chase_datamodule.train_indices.tolist() + chase_datamodule.val_indices.tolist(), fov = 30)
print("Dataset Statistics for 'train' split:")
print(stats)

Now that the datasets are in the right format, you can continue on the [U-Net pretraining notebook (2)](./02_pretrain_unet.ipynb)